# Programming Assignment 2 - My deep code analysis

## spark-newsgroups-random-forest



In [ ]:
import sys
import time
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import IDF, HashingTF, RegexTokenizer, StopWordsRemover
from pyspark.ml.tuning import ParamGridBuilder
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

DATA = "file:///s/chopin/n/under/C837364858/cs435/PA2/spark-newsgroups-random-forest"
OUT = f"{DATA}/PA2_output"


CATEGORIES = ["alt.atheism", "comp.graphics", "comp.os.ms-windows.misc", "comp.sys.ibm.pc.hardware",
              "comp.sys.mac.hardware", "comp.windows.x", "misc.forsale", "rec.autos", "rec.motorcycles",
              "rec.sport.baseball", "rec.sport.hockey", "sci.crypt", "sci.electronics", "sci.med", "sci.space",
              "soc.religion.christian", "talk.politics.guns", "talk.politics.mideast", "talk.politics.misc",
              "talk.religion.misc"]

spark = SparkSession.builder.appName("PA2").getOrCreate()


**The label table**

In [ ]:
labels = spark.createDataFrame(
    [(category, float(i)) for i, category in enumerate(CATEGORIES)],
    "category string, label double")
labels.show(20, truncate=False)

                                                
|category                |label|
|------------------------|-----|
|alt.atheism             |0.0  |
|comp.graphics           |1.0  |
|comp.os.ms-windows.misc |2.0  |
|comp.sys.ibm.pc.hardware|3.0  |
|comp.sys.mac.hardware   |4.0  |
|comp.windows.x          |5.0  |
|misc.forsale            |6.0  |
|rec.autos               |7.0  |
|rec.motorcycles         |8.0  |
|rec.sport.baseball      |9.0  |
|rec.sport.hockey        |10.0 |
|sci.crypt               |11.0 |
|sci.electronics         |12.0 |
|sci.med                 |13.0 |
|sci.space               |14.0 |
|soc.religion.christian  |15.0 |
|talk.politics.guns      |16.0 |
|talk.politics.mideast   |17.0 |
|talk.politics.misc      |18.0 |
|talk.religion.misc      |19.0 |



What's happening here:
- `enumerate(CATEGORIES)` pairs each name with its position, `(0, "alt.atheism")`, `(1, "comp.graphics")` and so on.
- `float(i)`: the PDF says `label` is a **Double**, and Spark's classifiers want double labels.
- The string `"category string, label double"` is the schema, meaning the column names and their types.

I see 20 rows, numbered 0 to 19. Later, when I read the files, I'll get the category name from the folder name and **join** it to this table to look up its number. That's the PDF's "preserve the label-to-category mapping".


**read the raw training files**

- `*/*` matches every category folder, and then every file inside each one.
- `wholetext=True` puts **one whole file in each row**. Without it, Spark gives one row per *line*, and a post would get split into dozens of rows.
- The whole post lands in a column called `value`.
- `F.input_file_name()` adds a `path` column with the full path of the file each row came from. That's our only clue to the category, because the category is the folder name.

In [ ]:
raw = spark.read.text(f"{DATA}/20news-bydate-train/*/*", wholetext=True).withColumn("path", F.input_file_name())
raw.count()
raw.show(3)



/>>> raw.count()                                                                 
11314                                                                           
/>>> raw.show(3)

|               value|                path|
|--------------------|--------------------|
|From: hamlet@stei...|file:///s/chopin/...|
|From: jeh@cmkrnl....|file:///s/chopin/...|
|From: mjp@watson....|file:///s/chopin/...|

only showing top 3 rows

11,314 is the exact number the PDF's Table 2 says for the training set.

**pull the category and file name out of the path**

In [ ]:

step = (raw
        .withColumn("category", F.regexp_extract("path", r"([^/]|)/([^/]|)$", 1))
        .withColumn("file_name", F.regexp_extract("path", r"([^/]|)/([^/]|)$", 2)))
step.select("path", "category", "file_name").show(3, truncate=False)


How to read that regex, `([^/]|)/([^/]|)$`:
- `[^/]|` means one or more characters that aren't `/`. So it's one piece of the path.
- The two `( )` groups are two pieces separated by a `/`.
- `$` means "at the very end of the string".
- Put together, it grabs **the last two pieces of the path**: `.../alt.atheism/49960` → group 1 = `alt.atheism`, group 2 = `49960`.
- The last argument, `1` or `2`, says which group to keep.

`withColumn` adds a new column and leaves the old ones alone, so  can watch the table grow. With `truncate=False`, now I see the full path next to its category and file name.

|path   |category  |file_name|
|-------|----------|---------|
|file:///s/chopin/n/under/C837364858/cs435/PA2/spark-newsgroups-random-forest/20news-bydate-train/rec.sport.hockey/53468  |rec.sport.hockey  |53468    |
|file:///s/chopin/n/under/C837364858/cs435/PA2/spark-newsgroups-random-forest/20news-bydate-train/sci.electronics/53569   |sci.electronics   |53569    |
|file:///s/chopin/n/under/C837364858/cs435/PA2/spark-newsgroups-random-forest/20news-bydate-train/talk.politics.guns/54684|talk.politics.guns|54684    |




**build the final columns from the PDF's Table 3**


In [ ]:

train_docs = (step
        .withColumn("doc_id", F.concat_ws("/", F.lit("train"), "category", "file_name"))
        .withColumn("source_split", F.lit("train"))
        .join(F.broadcast(labels), "category")
        .select("doc_id", F.col("value").alias("text"), "label", "category", "source_split"))
train_docs.printSchema()
train_docs.show(3)
train_docs.count()


/>>> train_docs.printSchema()
root

 |-- doc_id: string (nullable = false)

 |-- text: string (nullable = true)

 |-- label: double (nullable = true)

 |-- category: string (nullable = false)

 |-- source_split: string (nullable = false)
 

/>>> train_docs.show(3)


| doc_id|                text|label|          category|source_split|
|-----------|----------|-----|------------|------------|
|train/rec.sport.h...|From: hamlet@stei...| 10.0|  rec.sport.hockey|       train|
|train/sci.electro...|From: jeh@cmkrnl....| 12.0|   sci.electronics|       train|
|train/talk.politi...|From: mjp@watson....| 16.0|talk.politics.guns|       train|
only showing top 3 rows


What's new :
- `concat_ws("/", ...)` glues the pieces together with `/` to make the ID, like `train/rec.sport.hockey/53468`. The `train/` part keeps the ID unique even when we add the test set later.
- `F.lit("train")` makes a column that holds the same constant value in every row. That's our `source_split`.
- `.join(F.broadcast(labels), "category")` matches each row's category to the 20-row labels table and pulls in `label`. `broadcast` sends that tiny table to every worker so Spark doesn't have to shuffle the big table around.
- `.select(...)` keeps only the five columns we need and renames `value` to `text`.

Check `printSchema()` against Table 3. The types should be string, string, double, string, string. The count should still be 11,314. If it dropped, some category didn't match a label.

## For reusability, this can all be wrapped into a function that can accept train or test without repeating code. 

In [ ]:
def load(split):
    raw = spark.read.text(f"{DATA}/20news-bydate-{split}/*/*", wholetext=True).withColumn("path", F.input_file_name())
    return (raw
            .withColumn("category", F.regexp_extract("path", r"([^/]+)/([^/]+)$", 1))
            .withColumn("file_name", F.regexp_extract("path", r"([^/]+)/([^/]+)$", 2))
            .withColumn("doc_id", F.concat_ws("/", F.lit(split), "category", "file_name"))
            .withColumn("source_split", F.lit(split))
            .join(F.broadcast(labels), "category")
            .select("doc_id", F.col("value").alias("text"), "label", "category", "source_split")
            .cache())

official_train = load("train")
official_test = load("test")
print("official train:", official_train.count(), "| official test:", official_test.count())

**define the four feature stages (Task 2’s settings)**
count empty docs


settings:
- `pattern="\\W", gaps=True`: split wherever there's a **non-word** character (space, punctuation). `gaps=True` means the pattern describes the gaps *between* tokens, not the tokens themselves.
- `minTokenLength=2` drops single letters like "a" or "I".
- `toLowercase=True` makes "Hockey" and "hockey" the same word.
- Each stage's `outputCol` feeds the next stage's `inputCol`: text → words → kept → counts → features.

It's a function (not a plain list) for the same reason as Lab 5. Every call provides unfitted** stages, so the IDF fit on one dataset never gets reused by accident.



In [ ]:
def feature_stages():
    return [
        RegexTokenizer(inputCol="text", outputCol="words", pattern="\\W", gaps=True, minTokenLength=2, toLowercase=True),
        StopWordsRemover(inputCol="words", outputCol="kept"),
        HashingTF(inputCol="kept", outputCol="counts", numFeatures=4096, binary=False),
        IDF(inputCol="counts", outputCol="features", minDocFreq=2),
    ]

count the empty documents

PDF Section 3 says to keep posts that end up empty and report how many there are

In [ ]:
tokenize = Pipeline(stages=feature_stages()[:2]).fit(official_train)
for name, df in [("train", official_train), ("test", official_test)]:
    tokens = tokenize.transform(df)
    print(name, "| empty text:", df.filter(F.trim("text") == "").count(),
          "| no tokens:", tokens.filter(F.size("words") == 0).count(),
          "| no tokens after stop-word removal:", tokens.filter(F.size("kept") == 0).count())



**split the official train set 80/20 into train and validation**



In [ ]:
def rows_for(ids):
    return official_train.join(ids, "doc_id").repartition(8, "doc_id").sortWithinPartitions("doc_id").cache()

ids = official_train.select("doc_id").repartition(8, "doc_id").sortWithinPartitions("doc_id")
train_ids, validation_ids = ids.randomSplit([0.8, 0.2], seed=42)

train, validation = rows_for(train_ids), rows_for(validation_ids)
print("train:", train.count(), "| validation:", validation.count())


- `randomSplit([0.8, 0.2], seed=42)` randomly sends each row to one piece or the other, about 80% and 20%. `seed=42` makes it the same split every run, as the PDF requires.
- **Why split only the `doc_id`s and sort them first?** `randomSplit` depends on what order the rows are in. Spark doesn't guarantee row order between runs or machines. So we put the IDs into a fixed layout first, 8 partitions, sorted inside each one. That way seed 42 always gives the exact same split.
- `rows_for(ids)` then joins those IDs back to `official_train` to get the full rows (text, label, and so on), laid out the same fixed way.

Expect roughly **9,050 | 2,260**. The PDF says "approximately", so it won't be exactly 80/20. The two numbers should add up to 11,314.

9,108 + 2,206 = 11,314, so nothing was lost. That's about 80.5% / 19.5%, which counts as "approximately 80/20".

**run the checks the PDF asks for in Task 1**

- `intersect` keeps only the `doc_id`s that are in **both** sets. It should come out to **0**, which means no post is in both train and validation. That's the same overlap check as Lab 5.
- `distinct().count()` on `label` counts how many different categories show up. want **20 and 20**.
- The table counts posts per category in each set. The PDF warns that `randomSplit` isn't stratified, meaning it doesn't force every category to split exactly 80/20. So this table is  *inspecting* the class distribution. Each category's validation count should be roughly a quarter of its train count.
- `full_outer` + `fillna(0)`: if a category were missing from one side, it would still show up with a 0 instead of disappearing from the table.



In [ ]:

print("train/validation overlap:", train.select("doc_id").intersect(validation.select("doc_id")).count())
print("classes in train:", train.select("label").distinct().count(),
      "| classes in validation:", validation.select("label").distinct().count())

(train.groupBy("label", "category").count().withColumnRenamed("count", "train")
      .join(validation.groupBy("label", "category").count().withColumnRenamed("count", "validation"), ["label", "category"], "full_outer")
      .fillna(0)
      .orderBy("label")
      .show(20, truncate=False))


**save the split assignments (the last item in Task 1)**



In [ ]:
(train.select("doc_id", F.lit("train").alias("split"))
      .unionByName(validation.select("doc_id", F.lit("validation").alias("split")))
      .write.mode("overwrite").option("header", True).csv(f"{OUT}/split_assignments"))

- It makes a two-column table, `doc_id | split`, for every training post.
- `unionByName` stacks the train rows and validation rows into one table, matching the columns by name.
- `.write...csv(...)` saves it, so anyone can rebuild the exact split later. That's the PDF's "save document-to-split assignments for reproducibility".
- `mode("overwrite")` allows re-run it without an already exists error.

It'll make a **folder** called `PA2_output/split_assignments/` with `part-*.csv` files in it. That's normal for Spark, because each worker writes its own part. When it's done, **Task 1 is complete** and we start TF-IDF.

fit the TF-IDF pipeline on the training subset, then look inside one document

In [ ]:
feature_model = Pipeline(stages=feature_stages()).fit(train)

row = feature_model.transform(train.orderBy("doc_id").limit(1)).first()
print("document:", row["doc_id"])
print("excerpt:", row["text"][:300])
print("tokens:", row["words"][:40])
print("after stop-word removal:", row["kept"][:40])
print("vector size:", row["features"].size, "| nonzero entries:", row["features"].numNonzeros())


- `.fit(train)`: **this is the leakage rule in action.** IDF counts word frequencies across the 9,108 *training* posts only. Validation and test never touch it.
- `feature_model.transform(...)` runs one post through all four stages. Each stage adds its column: `words` → `kept` → `counts` → `features`.
- `orderBy("doc_id").limit(1)` picks the same post every time, so the output is reproducible. `.first()` brings back that one row as a Python object.
- `[:300]` and `[:40]` just trim the output so it fits on the screen.

These printouts are exactly the PDF's three Task 2 checkpoints: a short excerpt, tokens before and after stop-word removal, and the vector size plus its nonzero count. Compare the two token lists and 'll see words like "the", "is" and "from" disappear.

 **vector size: 4096** (that's `numFeatures`), and a nonzero count much smaller than that. That's the "sparse vector" idea from Section 2: one post only uses a small slice of the possible words.


- **Stop-word removal worked.** Compare the two lists: `from` (first) and `to` (after "related") are gone, and everything else shifted left. Spark's default English list only removes very common filler words, so words like `anything` stay.
- **Headers are in the tokens** (`mathew`, `mantis`, `gmt`, `1993`). reading the files as-is, which the TA said is the choice. One line in the report covers it.
- **4096 slots, only 725 nonzero.** That's sparse: about 18% filled, and this is a long FAQ post. A typical short post fills far fewer.

That's **Task 2 done.** (The PDF's TF-IDF validation by hand is optional.)

**Task 3, Step 12: the majority-class baseline**
```python
accuracy = MulticlassClassificationEvaluator(metricName="accuracy")
weighted_f1 = MulticlassClassificationEvaluator(metricName="f1")

majority = train.groupBy("label").count().orderBy(F.desc("count"), "label").first()["label"]
baseline = validation.withColumn("prediction", F.lit(majority))
print("majority category:", CATEGORIES[int(majority)], "| accuracy:", round(accuracy.evaluate(baseline), 4),
      "| weighted F1:", round(weighted_f1.evaluate(baseline), 4))
```
- The two **evaluators** compare the `label` and `prediction` columns and return one number each. Remember, `metricName="f1"` is *weighted* F1, as the PDF points out.
- `majority`: count the posts per label **in train** (never validation, that would be peeking), sort biggest first (`"label"` breaks ties), and take the top one.
- `baseline`: give **every** validation post that same guess using `F.lit`.



- **Accuracy 0.0499:** validation has 110 hockey posts out of 2,206, and 110 / 2206 = 0.0499. It only gets the hockey posts right. That's about 1 in 20, which is basically random guessing.
- **Weighted F1 0.0047:** every other class has F1 = 0, because it's never predicted, so its precision and recall are both 0. Hockey has recall 1.0 (it caught all of them) but precision 0.0499, so its F1 = 2·0.0499·1/(1.0499) ≈ 0.095. Weight that by hockey's share of validation (0.0499) and get **0.0047**.

That gap is why the PDF uses F1 and not just accuracy. Guessing one class looks "5% right", but F1 shows it's useless. **Any real model has to beat these two numbers.**

**Step 13: the initial Random Forest (Task 3's second baseline)**
```python
forest = RandomForestClassifier(
    labelCol="label", featuresCol="features",
    numTrees=20, maxDepth=5, featureSubsetStrategy="sqrt",
    minInstancesPerNode=1, maxBins=32, seed=42)

start = time.time()
model = Pipeline(stages=feature_stages() + [forest]).fit(train)
fit_seconds = time.time() - start

predictions = model.transform(validation).select("label", "prediction").cache()
print("forest accuracy:", round(accuracy.evaluate(predictions), 4),
      "| weighted F1:", round(weighted_f1.evaluate(predictions), 4), "| fit %.1fs" % fit_seconds)
```
- The settings are copied word for word from the PDF's Task 3 box.
- `feature_stages() + [forest]` is Python list addition: the four TF-IDF stages, then the forest, five stages in all. That's the "append a classifier to the feature pipeline" part.
- `.fit(train)` learns IDF *and* trains the forest, both on train only.
- `time.time()` before and after gives the **training time** the PDF asks to report.
- `.transform(validation)` produces predictions on posts the model never saw.

This one takes a little while. Expect something way above 0.05 but not amazing, because 20 shallow trees (depth 5) is a pretty weak forest.

**0.4266 accuracy and 0.4115 weighted F1**, about 8.5× better than guessing, in 4.5 seconds. That's the Task 3 results table:

| Model | Val. accuracy | Val. weighted F1 | Fit time |
|---|---|---|---|
| Majority class (rec.sport.hockey) | 0.0499 | 0.0047 | n/a |
| Random Forest (20 trees, depth 5) | 0.4266 | 0.4115 | 4.5 s |

Why only about 43%? Depth 5 means each tree asks at most 5 questions. Each question only looks at 64 random slots out of 4,096. And 20 classes is a lot to separate with that little. Task 4 is where we find out whether more trees and more depth help.

**Task 3 done.** On to tuning.

**Task 4, Step 14: cache the TF-IDF vectors and build the grid**
```python
grid = (ParamGridBuilder()
        .addGrid(forest.numTrees, [1, 5, 20, 50, 100])
        .addGrid(forest.maxDepth, [5, 10, 25])
        .addGrid(forest.minInstancesPerNode, [1, 5])
        .build())
print("configurations:", len(grid))

train_features = feature_model.transform(train).select("label", "features").cache()
validation_features = feature_model.transform(validation).select("label", "features").cache()
print("cached rows:", train_features.count(), validation_features.count())
```
- `ParamGridBuilder` makes every combination of the values from the PDF's table: 5 × 3 × 2 = **30**. Each entry in `grid` is a small dictionary of settings, like `{numTrees: 50, maxDepth: 10, minInstancesPerNode: 5}`.
- `feature_model` is the one  fit **on train** in Step 11. We reuse it to turn both sets into vectors once and cache them. The PDF says this is allowed because the feature settings never change. It means the 30 runs only retrain the forest, not the tokenizer.
- `.count()` forces Spark to actually compute and cache the vectors now.

 **30**, then **9108 2206**.

**Step 15: the tuning loop.** This is the heart of Task 4.
```python
results = []
for params in grid:
    start = time.time()
    fitted = forest.fit(train_features, params)
    fit_seconds = time.time() - start
    scored = fitted.transform(validation_features).select("label", "prediction").cache()
    r = {"numTrees": params[forest.numTrees],
         "maxDepth": params[forest.maxDepth],
         "minInstancesPerNode": params[forest.minInstancesPerNode],
         "accuracy": accuracy.evaluate(scored),
         "weighted_f1": weighted_f1.evaluate(scored),
         "fit_seconds": fit_seconds}
    results.append(r)
    print("numTrees %3d  maxDepth %2d  minInstancesPerNode %d  accuracy %.4f  weighted F1 %.4f  fit %.1fs"
          % (r["numTrees"], r["maxDepth"], r["minInstancesPerNode"], r["accuracy"], r["weighted_f1"], r["fit_seconds"]))
```

It follows the PDF's four steps for each configuration:
1. `forest.fit(train_features, params)` trains on the **same** training vectors every time. Passing `params` overrides the forest's settings just for this one fit.
2. `.transform(validation_features)` scores on the **same** validation vectors every time.
3. It records accuracy, weighted F1 and fit time in a dictionary and adds it to `results`.
4. Picking the winner comes in the next step.

All 30 done. The biggest forest reached **0.70 accuracy / 0.6958 weighted F1** in 34 s, compared with 0.41 F1 for the baseline forest. But it might not be the winner, so let the rule decide.

**Step 16: pick the configuration (the PDF's selection and tie-break rule)**
```python
best = min(results, key=lambda r: (-round(r["weighted_f1"], 4), r["maxDepth"], r["numTrees"], -r["minInstancesPerNode"]))
print("selected: numTrees", best["numTrees"], "| maxDepth", best["maxDepth"],
      "| minInstancesPerNode", best["minInstancesPerNode"], "| weighted F1 %.4f" % best["weighted_f1"])
```
This is the Lab 5 `min()` trick. Python's `min` sorts by the key tuple, first item first, so each item in the tuple is one rule from the PDF:
1. `-round(weighted_f1, 4)`: we want the **highest** F1, but `min` finds the lowest, so we flip the sign. Rounding to 4 places is what makes "ties to four decimal places" count as ties.
2. `maxDepth`: if F1 ties, the **lower** depth wins (smaller is picked first by `min`).
3. `numTrees`: next, **fewer** trees win.
4. `-minInstancesPerNode`: last, the **larger** value wins, so we flip the sign again.



**Selected: 100 trees, depth 25, minInstancesPerNode 1, validation weighted F1 0.7252.** That's the biggest, most flexible forest in the grid, and it beat its `minInstancesPerNode=5` twin (0.6958). Allowing tiny leaves helped here. It didn't overfit enough to hurt validation.

**Task 4 done.** Now the test set gets used, exactly once.

**Task 5, Step 17: refit on all of the official train set and score the test set**
```python
final_forest = RandomForestClassifier(labelCol="label", featuresCol="features",
                                      numTrees=best["numTrees"], maxDepth=best["maxDepth"],
                                      minInstancesPerNode=best["minInstancesPerNode"],
                                      featureSubsetStrategy="sqrt", maxBins=32, seed=42)
final_model = Pipeline(stages=feature_stages() + [final_forest]).fit(official_train)

test_predictions = final_model.transform(official_test).select("doc_id", "category", "label", "prediction").cache()
print("test accuracy:", round(accuracy.evaluate(test_predictions), 4))
print("weighted precision:", round(MulticlassClassificationEvaluator(metricName="weightedPrecision").evaluate(test_predictions), 4))
print("weighted recall:", round(MulticlassClassificationEvaluator(metricName="weightedRecall").evaluate(test_predictions), 4))
print("weighted F1:", round(weighted_f1.evaluate(test_predictions), 4))
```
These lines map onto the PDF's three Task 5 bullets:
- **"Rebuild the selected pipeline":** fresh `feature_stages()` plus a new forest built from `best`'s settings.
- **"Fit on the entire official training set"**: `.fit(official_train)`, all 11,314 posts including the old validation ones. Because the stages are fresh, this **refits IDF on the full training set** too, which is the second bullet.
- **"Evaluate on the official test set"**: `.transform(official_test)`. This is the first and only time the model sees those 7,532 posts.

Watch for **weighted recall = accuracy**. The PDF says they're always equal for single-label problems, and here's a live check of that claim.

 final test numbers:

| Metric | Test |
|---|---|
| Accuracy | 0.6348 |
| Weighted precision | 0.6398 |
| Weighted recall | **0.6348** (equals accuracy ✓) |
| Weighted F1 | 0.6261 |

There's the live check: weighted recall matches accuracy exactly, just like the PDF said it would.

**Why is test (0.63) lower than validation (0.73)?** Validation was a random slice of the same posts as train. The bydate test set holds posts written later in time. New threads, new people and new topics show up. Also, because I kept the headers, the model partly learned *who* posts where, like "mathew@mantis posts in atheism". Those names don't carry over to new authors. Report this as-is. The PDF says there's no minimum score.

**Step 18: the 20×20 confusion matrix**
```python
names = spark.createDataFrame([(float(i), c) for i, c in enumerate(CATEGORIES)], "prediction double, predicted string")

(test_predictions.join(names, "prediction")
                 .groupBy("label", "category").pivot("predicted", CATEGORIES).count().fillna(0)
                 .orderBy("label")
                 .show(20, truncate=False))
```
- `names` is like `labels`, but for the prediction column. It turns `10.0` into `rec.sport.hockey` so the columns get real names.
- `groupBy("label", "category")` makes **one row per true category**, which is the PDF's "true labels as rows".
- `.pivot("predicted", CATEGORIES)` makes **one column per predicted category**. Passing `CATEGORIES` fixes the order to match the rows and keeps all 20 columns even if one is never predicted.
- `.count().fillna(0)` puts the number of posts in each cell, with 0 where there are none.

The **diagonal** (row = column) counts the posts it got right. Everything off the diagonal is a mistake. When  look at it, find the biggest off-diagonal numbers. Which categories does it confuse? (I'd bet on the `comp.*` ones and on religion/atheism.) It's a wide table, so make the terminal wide or zoom out before  screenshot it.

That's the labeled 20×20 matrix. Here's what it shows. The biggest off-diagonal cells (where the row is the truth and the column is the guess):

| True category | Predicted as | Count |
|---|---|---|
| talk.politics.misc | talk.politics.guns | **77** |
| comp.windows.x | comp.os.ms-windows.misc | **71** |
| talk.religion.misc | soc.religion.christian | **64** |
| alt.atheism | soc.religion.christian | **61** |
| sci.med | sci.electronics | **57** |
| comp.sys.ibm.pc.hardware | comp.os.ms-windows.misc | **47** |

The story for the report: **the mistakes happen between neighboring topics.** Politics gets confused with politics, religion with religion, and computers with computers. That's because those groups share vocabulary ("god", "bible", "windows", "drivers", "government"). The "misc" catch-all groups suffer most. talk.religion.misc only got 75 right, and it sent almost as many (64) to christian. sci.electronics (125 right) is spread across almost every column. Meanwhile, distinctive topics like soc.religion.christian (348), rec.motorcycles (332) and sci.crypt (330) are strong.

**Step 19: per-class precision, recall, F1, and macro F1 (the last piece)**
```python
counts = test_predictions.groupBy("label", "prediction").count()
per_class = (labels
    .join(counts.filter(F.col("label") == F.col("prediction")).select("label", F.col("count").alias("tp")), "label", "left")
    .join(counts.groupBy("label").agg(F.sum("count").alias("support")), "label", "left")
    .join(counts.groupBy("prediction").agg(F.sum("count").alias("predicted")).withColumnRenamed("prediction", "label"), "label", "left")
    .fillna(0, ["tp", "support", "predicted"])
    .withColumn("precision", F.when(F.col("predicted") > 0, F.col("tp") / F.col("predicted")).otherwise(0.0))
    .withColumn("recall", F.when(F.col("support") > 0, F.col("tp") / F.col("support")).otherwise(0.0))
    .withColumn("f1", F.when(F.col("precision") + F.col("recall") > 0,
                             2 * F.col("precision") * F.col("recall") / (F.col("precision") + F.col("recall"))).otherwise(0.0))
    .orderBy("label"))
per_class.select("label", "category", F.round("precision", 4).alias("precision"),
                 F.round("recall", 4).alias("recall"), F.round("f1", 4).alias("f1")).show(20, truncate=False)
print("macro F1:", round(per_class.agg(F.avg("f1")).first()[0], 4))
```
This builds the PDF formulas straight from the confusion matrix counts:
- `counts` gives one row per (true, predicted) pair. That's the matrix in long form.
- **tp** = rows where label == prediction, which is the **diagonal**.
- **support** = the total for each true label, which is the **row sum** (TP + FN).
- **predicted** = the total for each prediction, which is the **column sum** (TP + FP).
- precision = tp/predicted and recall = tp/support. `F.when(... > 0 ... ).otherwise(0.0)` is the PDF's rule: "zero when the denominator is zero".
- f1 = 2PR / (P+R).
- **Macro F1** = `avg("f1")`, a plain average over all 20 classes, every class counted equally.

Prediction: will macro F1 come out a bit lower or higher than the weighted F1 (0.6261)? Think about which classes are small *and* weak.

**Macro F1 = 0.6215.** That's a little below weighted F1 (0.6261), and here's why. Macro counts every class equally, so the weak classes pull it down at full strength: talk.religion.misc (0.43), sci.med (0.46) and sci.electronics (0.32). Weighted F1 shrinks talk.religion.misc's vote because it's the smallest class. The gap is small because most classes are about the same size.

Two more patterns worth a sentence in report:
- **High precision, low recall** for talk.religion.misc (0.75 / 0.30) and talk.politics.misc (0.75 / 0.43). The model rarely picks them, but it's usually right when it does. Most of their posts get pulled into the bigger neighbor (christian, guns).
- **sci.electronics is weak on both** (0.32 / 0.32). Its vocabulary overlaps with the comp.* groups, autos and med.

**all five tasks done.** 


For the report: the Task 3 table, the [tuning table](/mnt/project-files/pa2/tuning_results.md) with the runtime discussion points, the selected configuration, the final metrics, the per-class table, and the confusion matrix.


## Setup

I used the 20 Newsgroups dataset with its partition unchanged. It had 11,314 training documents and 7,532 test documents across 20 categories. Each post was read as one whole-text row into a Spark DataFrame with the columns doc_id, text, label, category and source_split. Labels 0 to 19 follow the alphabetical order of the category folders. I read each file as-is, keeping the e-mail headers, since the assignment leaves the cleaning choice to the student. No documents were empty in either partition, whether checked as raw text, after tokenization, or after stop-word removal, so no documents received zero vectors. Seed 42 was used everywhere it is supported, and every document's split assignment was saved to `PA2_output/split_assignments`.

## Task 1: partition

| Set | Documents |
|---|---:|
| Official train | 11,314 |
| Train subset (≈80%) | 9,108 |
| Validation subset (≈20%) | 2,206 |
| Official test | 7,532 |

I split the official training set with randomSplit([0.8, 0.2], seed=42) after putting the document IDs into a fixed partition layout and order, so the split is reproducible. This produced 9,108 training and 2,206 validation documents (about 80.5% / 19.5%). The two subsets share no document IDs, and both contain all 20 classes. Because randomSplit is not stratified, the validation share per class varies somewhat, from about 17% for talk.politics.misc (78 of 465) to about 22% for alt.atheism (105 of 480). The variation is small, and every class is well represented in both subsets. The full per-class distribution table is in PA2_outputs.txt.

## Task 2: feature extraction check (train/alt.atheism/49960)

The feature pipeline is RegexTokenizer (lowercase, splitting on non-word characters, minimum token length 2) → StopWordsRemover (default English list) → HashingTF (4,096 features, raw counts) → IDF (minDocFreq 2). IDF was fit on the 9,108-document training subset only and then applied to the validation subset, so no validation or test documents influenced the IDF weights. For the first training document, stop-word removal dropped common words such as "from" and "to". Its TF-IDF vector has 4,096 dimensions but only 725 nonzero entries, which shows how sparse the representation is: each document uses only a small fraction of the hashed vocabulary.

## Task 3: baselines (validation set)

| Model | Accuracy | Weighted F1 | Fit time |
|---|---:|---:|---:|
| Majority class (rec.sport.hockey) | 0.0499 | 0.0047 | n/a |
| Random Forest (20 trees, depth 5, minInstancesPerNode 1) | 0.4266 | 0.4115 | 4.5 s |

The majority-class baseline always predicts rec.sport.hockey, the most common category in the training subset. It is correct only for the 110 hockey posts in validation (accuracy 0.0499), and its weighted F1 is near zero because every other class has an F1 of 0. The initial Random Forest reaches 0.4266 accuracy and 0.4115 weighted F1 in 4.5 seconds, roughly 8.5 times the baseline accuracy. That shows the TF-IDF features carry real topic information. Still, 20 trees of depth 5 is a weak model for 20 classes, because each tree can ask at most five questions and each split only considers √4096 = 64 random features.

## Task 4: tuning (validation set)

| numTrees | maxDepth | minInstancesPerNode | Accuracy | Weighted F1 | Fit time (s) |
|---:|---:|---:|---:|---:|---:|
| 1 | 5 | 1 | 0.1238 | 0.0808 | 1.4 |
| 1 | 5 | 5 | 0.1238 | 0.0808 | 1.3 |
| 1 | 10 | 1 | 0.2049 | 0.1730 | 1.9 |
| 1 | 10 | 5 | 0.1868 | 0.1719 | 1.9 |
| 1 | 25 | 1 | 0.2919 | 0.3005 | 3.9 |
| 1 | 25 | 5 | 0.2906 | 0.3070 | 3.8 |
| 5 | 5 | 1 | 0.2226 | 0.2049 | 1.4 |
| 5 | 5 | 5 | 0.2502 | 0.2324 | 1.4 |
| 5 | 10 | 1 | 0.3577 | 0.3638 | 2.0 |
| 5 | 10 | 5 | 0.3803 | 0.3695 | 2.1 |
| 5 | 25 | 1 | 0.5009 | 0.5015 | 5.0 |
| 5 | 25 | 5 | 0.5009 | 0.4965 | 4.8 |
| 20 | 5 | 1 | 0.4266 | 0.4115 | 1.4 |
| 20 | 5 | 5 | 0.4461 | 0.4349 | 1.4 |
| 20 | 10 | 1 | 0.5295 | 0.5189 | 3.0 |
| 20 | 10 | 5 | 0.5286 | 0.5154 | 2.7 |
| 20 | 25 | 1 | 0.6523 | 0.6463 | 10.1 |
| 20 | 25 | 5 | 0.6405 | 0.6374 | 8.6 |
| 50 | 5 | 1 | 0.4900 | 0.4732 | 1.9 |
| 50 | 5 | 5 | 0.5227 | 0.5070 | 1.9 |
| 50 | 10 | 1 | 0.5884 | 0.5770 | 4.4 |
| 50 | 10 | 5 | 0.6015 | 0.5973 | 4.4 |
| 50 | 25 | 1 | 0.7126 | 0.7070 | 20.6 |
| 50 | 25 | 5 | 0.6886 | 0.6841 | 17.1 |
| 100 | 5 | 1 | 0.5535 | 0.5328 | 2.7 |
| 100 | 5 | 5 | 0.5589 | 0.5411 | 2.9 |
| 100 | 10 | 1 | 0.6197 | 0.6089 | 7.0 |
| 100 | 10 | 5 | 0.6129 | 0.5993 | 7.2 |
| **100** | **25** | **1** | **0.7303** | **0.7252** | **40.4** |
| 100 | 25 | 5 | 0.7017 | 0.6958 | 33.6 |

**Selected configuration:** numTrees = 100, maxDepth = 25, minInstancesPerNode = 1 (featureSubsetStrategy = sqrt, maxBins = 32, seed = 42). It had the highest validation weighted F1 (0.7252), so the tie-break rule wasn't needed.

### Accuracy–runtime tradeoffs

Tree depth had the largest effect on quality. With 100 trees, increasing maxDepth from 5 to 25 raised validation weighted F1 from 0.533 to 0.725. Shallow trees cannot form enough splits to separate 20 sparse, high-dimensional classes. Adding trees also helped, but with clearly diminishing returns. At depth 25, weighted F1 rose from 0.30 with one tree to 0.50, 0.65, 0.71 and 0.73 with 5, 20, 50 and 100 trees.

Runtime grows with the product of trees and depth. Every depth-5 forest trained in about 1.3 to 2.9 seconds regardless of tree count, while the depth-25 forests grew from about 4 seconds (one tree) to 40.4 seconds (100 trees). The last step is the clearest tradeoff: going from 50 to 100 trees at depth 25 doubled the fit time (20.6 s to 40.4 s) but gained only 0.018 in weighted F1. If training time mattered, the 50-tree, depth-25 forest would give about 97% of the selected model's F1 in about half the time. minInstancesPerNode = 5 slightly helped the shallow forests, but it lowered weighted F1 for every depth-25 forest with 5 or more trees. Those deep trees benefit from small leaves rather than from extra regularization. I followed the assignment's rule and selected the configuration with the highest validation weighted F1, which was also the most expensive one.

## Task 5: final model (official test set)

Refit on all 11,314 official training documents (IDF refit too), then evaluated once on the 7,532 test documents.

| Metric | Test |
|---|---:|
| Accuracy | 0.6348 |
| Weighted precision | 0.6398 |
| Weighted recall | 0.6348 |
| Weighted F1 | 0.6261 |
| Macro F1 | 0.6215 |

### Per-class metrics

| Label | Category | Support | Precision | Recall | F1 |
|---:|---|---:|---:|---:|---:|
| 0 | alt.atheism | 319 | 0.7054 | 0.5329 | 0.6071 |
| 1 | comp.graphics | 389 | 0.5398 | 0.4884 | 0.5128 |
| 2 | comp.os.ms-windows.misc | 394 | 0.5818 | 0.7132 | 0.6408 |
| 3 | comp.sys.ibm.pc.hardware | 392 | 0.5829 | 0.5383 | 0.5597 |
| 4 | comp.sys.mac.hardware | 385 | 0.6082 | 0.5766 | 0.5920 |
| 5 | comp.windows.x | 395 | 0.6045 | 0.5418 | 0.5714 |
| 6 | misc.forsale | 390 | 0.6329 | 0.8000 | 0.7067 |
| 7 | rec.autos | 396 | 0.6765 | 0.6338 | 0.6545 |
| 8 | rec.motorcycles | 398 | 0.6734 | 0.8342 | 0.7452 |
| 9 | rec.sport.baseball | 397 | 0.6030 | 0.7078 | 0.6512 |
| 10 | rec.sport.hockey | 399 | 0.7099 | 0.8095 | 0.7564 |
| 11 | sci.crypt | 396 | 0.7449 | 0.8333 | 0.7867 |
| 12 | sci.electronics | 393 | 0.3213 | 0.3181 | 0.3197 |
| 13 | sci.med | 396 | 0.6266 | 0.3687 | 0.4642 |
| 14 | sci.space | 394 | 0.6906 | 0.7817 | 0.7333 |
| 15 | soc.religion.christian | 398 | 0.6421 | 0.8744 | 0.7404 |
| 16 | talk.politics.guns | 364 | 0.5833 | 0.7308 | 0.6488 |
| 17 | talk.politics.mideast | 376 | 0.8484 | 0.6995 | 0.7668 |
| 18 | talk.politics.misc | 310 | 0.7472 | 0.4290 | 0.5451 |
| 19 | talk.religion.misc | 251 | 0.7500 | 0.2988 | 0.4274 |

Support is each row's total in the confusion matrix (true documents per class).

### Confusion matrix

![alt text](image-1.png)

Largest confusions:

| True | Predicted as | Count |
|---|---|---:|
| talk.politics.misc | talk.politics.guns | 77 |
| comp.windows.x | comp.os.ms-windows.misc | 71 |
| talk.religion.misc | soc.religion.christian | 64 |
| alt.atheism | soc.religion.christian | 61 |
| sci.med | sci.electronics | 57 |
| comp.sys.ibm.pc.hardware | comp.os.ms-windows.misc | 47 |

### Interpretation

The final model correctly classifies 63.5% of the test documents, with a weighted F1 of 0.626. Weighted recall equals accuracy (0.6348), as expected for single-label multiclass classification. Macro F1 (0.6215) is slightly below weighted F1 (0.6261). Macro F1 gives every class equal weight, and the two smallest test classes, talk.religion.misc and talk.politics.misc, are also among the weakest.

The confusion matrix shows that most errors happen between neighboring topics that share vocabulary. Politics is confused with politics (77 talk.politics.misc posts predicted as talk.politics.guns), religion with religion (talk.religion.misc and alt.atheism both lose many posts to soc.religion.christian), and computing with computing (71 comp.windows.x posts predicted as comp.os.ms-windows.misc). The broad "misc" categories show high precision but low recall: talk.religion.misc has 0.75 precision but 0.30 recall, and talk.politics.misc has 0.75 and 0.43. The model rarely predicts these classes and is usually right when it does, but most of their documents are absorbed by a larger, more specific neighbor. sci.electronics is the weakest class on both measures (F1 0.32), with errors spread across the comp.*, rec.autos and sci.med columns, consistent with its overlapping technical vocabulary. Categories with distinctive vocabulary perform best, such as sci.crypt (F1 0.79), talk.politics.mideast (0.77) and rec.sport.hockey (0.76).

Test performance (weighted F1 0.626) is noticeably lower than the best validation score (0.725). The validation subset was drawn at random from the same time period as the training data, while the bydate test set contains posts written later, with new threads, topics and authors. Because the e-mail headers were kept, part of what the model learned is tied to specific authors and organizations, which do not carry over to new posters. The test set was used only once, after model selection, so this gap is an honest estimate of how the model generalizes to future posts.


# PA2: Document Classification with Random Forest

## Task 3: Baseline results (validation set)

| Model | Accuracy | Weighted F1 | Fit time |
|---|---:|---:|---:|
| Majority class (rec.sport.hockey) | 0.0499 | 0.0047 | n/a |
| Random Forest (20 trees, depth 5, minInstancesPerNode 1) | 0.4266 | 0.4115 | 4.5 s |

## Task 4: Hyperparameter tuning (validation set)

| numTrees | maxDepth | minInstancesPerNode | Accuracy | Weighted F1 | Fit time (s) |
|---:|---:|---:|---:|---:|---:|
| 1 | 5 | 1 | 0.1238 | 0.0808 | 1.4 |
| 1 | 5 | 5 | 0.1238 | 0.0808 | 1.3 |
| 1 | 10 | 1 | 0.2049 | 0.1730 | 1.9 |
| 1 | 10 | 5 | 0.1868 | 0.1719 | 1.9 |
| 1 | 25 | 1 | 0.2919 | 0.3005 | 3.9 |
| 1 | 25 | 5 | 0.2906 | 0.3070 | 3.8 |
| 5 | 5 | 1 | 0.2226 | 0.2049 | 1.4 |
| 5 | 5 | 5 | 0.2502 | 0.2324 | 1.4 |
| 5 | 10 | 1 | 0.3577 | 0.3638 | 2.0 |
| 5 | 10 | 5 | 0.3803 | 0.3695 | 2.1 |
| 5 | 25 | 1 | 0.5009 | 0.5015 | 5.0 |
| 5 | 25 | 5 | 0.5009 | 0.4965 | 4.8 |
| 20 | 5 | 1 | 0.4266 | 0.4115 | 1.4 |
| 20 | 5 | 5 | 0.4461 | 0.4349 | 1.4 |
| 20 | 10 | 1 | 0.5295 | 0.5189 | 3.0 |
| 20 | 10 | 5 | 0.5286 | 0.5154 | 2.7 |
| 20 | 25 | 1 | 0.6523 | 0.6463 | 10.1 |
| 20 | 25 | 5 | 0.6405 | 0.6374 | 8.6 |
| 50 | 5 | 1 | 0.4900 | 0.4732 | 1.9 |
| 50 | 5 | 5 | 0.5227 | 0.5070 | 1.9 |
| 50 | 10 | 1 | 0.5884 | 0.5770 | 4.4 |
| 50 | 10 | 5 | 0.6015 | 0.5973 | 4.4 |
| 50 | 25 | 1 | 0.7126 | 0.7070 | 20.6 |
| 50 | 25 | 5 | 0.6886 | 0.6841 | 17.1 |
| 100 | 5 | 1 | 0.5535 | 0.5328 | 2.7 |
| 100 | 5 | 5 | 0.5589 | 0.5411 | 2.9 |
| 100 | 10 | 1 | 0.6197 | 0.6089 | 7.0 |
| 100 | 10 | 5 | 0.6129 | 0.5993 | 7.2 |
| **100** | **25** | **1** | **0.7303** | **0.7252** | **40.4** |
| 100 | 25 | 5 | 0.7017 | 0.6958 | 33.6 |

**Selected configuration:** numTrees = 100, maxDepth = 25, minInstancesPerNode = 1 (featureSubsetStrategy = sqrt, maxBins = 32, seed = 42). Highest validation weighted F1 (0.7252).

### Accuracy–runtime tradeoffs

Tree depth had the largest effect on quality. With 100 trees, increasing maxDepth from 5 to 25 raised validation weighted F1 from 0.533 to 0.725. Adding trees also helped, but with diminishing returns: at depth 25, weighted F1 went from 0.30 with one tree to 0.50, 0.65, 0.71 and 0.73 with 5, 20, 50 and 100 trees.

Runtime grows with both the number of trees and the depth. The depth-5 forests all trained in about 1.3 to 2.9 seconds, while the depth-25 forests went from about 4 seconds (one tree) to 40.4 seconds (100 trees). Going from 50 to 100 trees at depth 25 doubled the fit time (20.6 s to 40.4 s) for only 0.018 more weighted F1, so the 50-tree forest would be a reasonable choice if training time mattered. Setting minInstancesPerNode to 5 helped the shallow forests slightly but lowered F1 for the deep forests with 5 or more trees. I picked the configuration with the highest validation weighted F1, which was also the slowest one.

## Task 5: Final model (official test set)

Fit on the full official training set (11,314 documents, IDF refit), evaluated once on the test set (7,532 documents).

### Final model accuracy

**Accuracy: 0.6348**

| Metric | Test |
|---|---:|
| Weighted precision | 0.6398 |
| Weighted recall | 0.6348 |
| Weighted F1 | 0.6261 |
| Macro F1 | 0.6215 |

### Precision, recall, F1 (per class)

| Label | Category | Support | Precision | Recall | F1 |
|---:|---|---:|---:|---:|---:|
| 0 | alt.atheism | 319 | 0.7054 | 0.5329 | 0.6071 |
| 1 | comp.graphics | 389 | 0.5398 | 0.4884 | 0.5128 |
| 2 | comp.os.ms-windows.misc | 394 | 0.5818 | 0.7132 | 0.6408 |
| 3 | comp.sys.ibm.pc.hardware | 392 | 0.5829 | 0.5383 | 0.5597 |
| 4 | comp.sys.mac.hardware | 385 | 0.6082 | 0.5766 | 0.5920 |
| 5 | comp.windows.x | 395 | 0.6045 | 0.5418 | 0.5714 |
| 6 | misc.forsale | 390 | 0.6329 | 0.8000 | 0.7067 |
| 7 | rec.autos | 396 | 0.6765 | 0.6338 | 0.6545 |
| 8 | rec.motorcycles | 398 | 0.6734 | 0.8342 | 0.7452 |
| 9 | rec.sport.baseball | 397 | 0.6030 | 0.7078 | 0.6512 |
| 10 | rec.sport.hockey | 399 | 0.7099 | 0.8095 | 0.7564 |
| 11 | sci.crypt | 396 | 0.7449 | 0.8333 | 0.7867 |
| 12 | sci.electronics | 393 | 0.3213 | 0.3181 | 0.3197 |
| 13 | sci.med | 396 | 0.6266 | 0.3687 | 0.4642 |
| 14 | sci.space | 394 | 0.6906 | 0.7817 | 0.7333 |
| 15 | soc.religion.christian | 398 | 0.6421 | 0.8744 | 0.7404 |
| 16 | talk.politics.guns | 364 | 0.5833 | 0.7308 | 0.6488 |
| 17 | talk.politics.mideast | 376 | 0.8484 | 0.6995 | 0.7668 |
| 18 | talk.politics.misc | 310 | 0.7472 | 0.4290 | 0.5451 |
| 19 | talk.religion.misc | 251 | 0.7500 | 0.2988 | 0.4274 |

### Confusion matrix
![alt text](image-2.png)
Rows are the true category, columns are the predicted category (column numbers match the row labels).

| True \ Pred | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 | 13 | 14 | 15 | 16 | 17 | 18 | 19 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 0 alt.atheism | 170 | 2 | 0 | 0 | 4 | 3 | 4 | 2 | 8 | 7 | 2 | 7 | 6 | 5 | 8 | 61 | 11 | 5 | 1 | 13 |
| 1 comp.graphics | 1 | 190 | 29 | 19 | 15 | 26 | 8 | 3 | 17 | 10 | 7 | 5 | 18 | 10 | 13 | 5 | 8 | 4 | 1 | 0 |
| 2 comp.os.ms-windows.misc | 0 | 12 | 281 | 28 | 9 | 15 | 10 | 6 | 3 | 3 | 0 | 2 | 12 | 3 | 6 | 0 | 1 | 2 | 0 | 1 |
| 3 comp.sys.ibm.pc.hardware | 1 | 24 | 47 | 211 | 14 | 9 | 17 | 13 | 1 | 12 | 6 | 5 | 19 | 0 | 9 | 2 | 0 | 1 | 1 | 0 |
| 4 comp.sys.mac.hardware | 1 | 11 | 14 | 25 | 222 | 18 | 18 | 7 | 12 | 15 | 3 | 3 | 20 | 4 | 6 | 2 | 3 | 0 | 1 | 0 |
| 5 comp.windows.x | 0 | 25 | 71 | 11 | 14 | 214 | 4 | 7 | 6 | 3 | 4 | 2 | 9 | 1 | 12 | 2 | 3 | 3 | 4 | 0 |
| 6 misc.forsale | 1 | 7 | 7 | 14 | 11 | 6 | 312 | 4 | 4 | 3 | 1 | 1 | 9 | 2 | 6 | 0 | 1 | 1 | 0 | 0 |
| 7 rec.autos | 2 | 10 | 2 | 4 | 5 | 3 | 16 | 251 | 16 | 13 | 14 | 2 | 22 | 6 | 7 | 0 | 18 | 1 | 2 | 2 |
| 8 rec.motorcycles | 1 | 2 | 0 | 3 | 5 | 1 | 7 | 13 | 332 | 6 | 4 | 1 | 8 | 5 | 5 | 1 | 2 | 1 | 1 | 0 |
| 9 rec.sport.baseball | 2 | 8 | 1 | 1 | 7 | 6 | 12 | 9 | 8 | 281 | 28 | 10 | 12 | 2 | 2 | 4 | 1 | 1 | 1 | 1 |
| 10 rec.sport.hockey | 0 | 2 | 1 | 2 | 6 | 2 | 14 | 0 | 7 | 23 | 323 | 2 | 4 | 2 | 4 | 2 | 3 | 2 | 0 | 0 |
| 11 sci.crypt | 1 | 2 | 1 | 3 | 2 | 1 | 4 | 5 | 2 | 7 | 2 | 330 | 10 | 4 | 6 | 3 | 7 | 0 | 5 | 1 |
| 12 sci.electronics | 3 | 26 | 13 | 28 | 23 | 15 | 9 | 26 | 19 | 21 | 12 | 24 | 125 | 11 | 16 | 4 | 10 | 3 | 5 | 0 |
| 13 sci.med | 13 | 9 | 2 | 6 | 14 | 15 | 19 | 3 | 39 | 20 | 7 | 5 | 57 | 146 | 12 | 18 | 5 | 2 | 4 | 0 |
| 14 sci.space | 0 | 3 | 2 | 1 | 1 | 8 | 10 | 2 | 2 | 6 | 9 | 5 | 13 | 8 | 308 | 1 | 9 | 2 | 4 | 0 |
| 15 soc.religion.christian | 3 | 2 | 1 | 1 | 2 | 4 | 5 | 0 | 2 | 5 | 1 | 0 | 9 | 1 | 3 | 348 | 1 | 4 | 1 | 5 |
| 16 talk.politics.guns | 0 | 4 | 4 | 1 | 2 | 3 | 13 | 3 | 6 | 6 | 12 | 17 | 7 | 3 | 5 | 2 | 266 | 3 | 7 | 0 |
| 17 talk.politics.mideast | 15 | 6 | 1 | 0 | 3 | 4 | 1 | 2 | 2 | 11 | 8 | 7 | 13 | 4 | 2 | 14 | 13 | 263 | 5 | 2 |
| 18 talk.politics.misc | 2 | 4 | 4 | 1 | 5 | 1 | 6 | 9 | 4 | 6 | 10 | 9 | 10 | 5 | 8 | 9 | 77 | 7 | 133 | 0 |
| 19 talk.religion.misc | 25 | 3 | 2 | 3 | 1 | 0 | 4 | 6 | 3 | 8 | 2 | 6 | 6 | 11 | 8 | 64 | 17 | 5 | 2 | 75 |
